In [1]:
from git import Repo

from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_groq import ChatGroq

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

from dotenv import load_dotenv
import os

#Clone the Langchain Git repo

In [2]:
!mkdir test_repo


In [3]:
# Cell 5 - Clone Repo
repo_path = "test_repo/"
repo = Repo.clone_from(
    "https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI", 
    to_path=repo_path
)

**LOAD THE DATA FROM THE FILE**

In [4]:
%pwd

'/Users/pranay/Desktop/final pro/Real-Time-Scource-Code-Analyzer/research'

In [5]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*",
    suffixes=[".py"],
    parser=LanguageParser(language="python", parser_threshold=500)
)

documents = loader.load()

print("Total files:", len(documents))

Total files: 7


In [6]:
documents = loader.load()

In [7]:
documents[0]

Document(page_content="from setuptools import find_packages, setup\n\nsetup(\n    name = 'Generative AI Project',\n    version= '0.0.0',\n    author= 'Bappy Ahmed',\n    author_email= 'entbappy73@gmail.com',\n    packages= find_packages(),\n    install_requires = []\n\n)", metadata={'source': 'test_repo/setup.py', 'language': 'python'})

In [8]:
len(documents)

7

**Split the document into chucks**

In [9]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(
    language="python",
    chunk_size=500,
    chunk_overlap=20
)

texts = documents_splitter.split_documents(documents)

print("Total chunks:", len(texts))

Total chunks: 13


In [12]:
len(texts)

13

Step 6 : Download the ggroqEmbeddings

In [11]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("API Loaded:", GROQ_API_KEY is not None)

API Loaded: True


In [13]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10755.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
vectordb = Chroma.from_documents(
    texts,
    embedding=embeddings,
    persist_directory="./db"
)

vectordb.persist()

/opt/anaconda3/envs/llmapp/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  warn_deprecated(


In [19]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",   # you can change to llama3-70b-8192
    groq_api_key=GROQ_API_KEY,
    temperature=0
)

In [20]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

In [ ]:
qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 20}
    ),
    memory=memory
)

In [22]:
question = "what is download_hugging_face_embeddings function doing?"

result = qa(question)

print(result["answer"])

The `download_hugging_face_embeddings` function is downloading embeddings from Hugging Face. 

Here's a step-by-step breakdown of what it does:

1. It imports the `HuggingFaceEmbeddings` class from an unspecified module (likely a custom module or a library that provides Hugging Face embeddings functionality).

2. It creates an instance of the `HuggingFaceEmbeddings` class, passing the model name `'sentence-transformers/all-MiniLM-L6-v2'` as an argument. This model returns 384-dimensional embeddings.

3. It returns the instance of the `HuggingFaceEmbeddings` class, which presumably contains the downloaded embeddings.

In essence, this function is downloading a pre-trained model from Hugging Face's model hub and making it available for use in the script. The specific model used here is a sentence transformer model called `all-MiniLM-L6-v2`, which is a smaller version of the original model and is suitable for smaller-scale applications.
